<div style="display: flex; align-items: center;">
    <h1>Tutorial on hybrid crop modelling with diffWOFOST</h1>
    <img src="https://raw.githubusercontent.com/WUR-AI/diffWOFOST/refs/heads/main/docs/logo/diffwofost.png" width="150" style="margin-left: 20px;">
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ronvree/diffwofost-tutorial/blob/master/hybrid_stress_correction.ipynb)

In this tutorial we will build a **hybrid crop model** by
plugging a small neural network into WOFOST72 and training
it end-to-end. The notebook
uses the [diffwofost](https://github.com/WUR-AI/diffWOFOST) Python package and a public
field-trial dataset.

By the end you will have:

- explored a real field-trial dataset (potato, two Dutch sites, 168 plot-years (174 before dropping Shadow cultivar));
- replaced WOFOST's standard evapotranspiration block with a learnable stress
  factor (`RFTRA`) produced by a neural network;
- compared the hybrid against an uncalibrated WOFOST72_PP reference *and*
  against a pure-ML LSTM with no physics in the loop;
- seen a teaser of what else differentiability buys you (parameter sensitivities
  for free).

> **Citations.** The field data is from Ten Den et al. (2024), Harvard
> Dataverse [doi:10.7910/DVN/1LC6W7](https://doi.org/10.7910/DVN/1LC6W7),
> CC BY-NC-SA 4.0. The diffwofost engine and this tutorial are
> EUPL-1.1. Pretrained models in this notebook are derivative works of
> the field data and are therefore also released under CC BY-NC-SA 4.0.


## 1. The big picture: why hybrid modelling?

Mechanistic crop models like **WOFOST** encode decades of agronomic knowledge as
ODEs (Ordinary Differential Equation): photosynthesis, partitioning, water balance, phenology. They are
*interpretable*, *physically grounded*, and *generalise* to weather and
management regimes outside their calibration data — but they have to make
simplifying assumptions, and the assumptions sometimes break.

Pure **data-driven** models (an LSTM that maps weather + soil → yield) can
absorb whatever structure is in the data, but they have **no inductive bias**:
nothing tells them carbon is conserved, that leaves grow before tubers, or that
photosynthesis caps at light saturation. They overfit small datasets badly and
extrapolate poorly outside the training distribution.

**Hybrid models** keep the physics where it's well-understood and let a neural
network fill in the parts that are poorly modelled. Concretely, this tutorial:

| Component | Source |
|-----------|--------|
| Phenology (DVS, TSUM) | physical (WOFOST) |
| Carbon partitioning to leaves/stems/tubers | physical (WOFOST) |
| Soil water balance (Potential Production variant) | physical (WOFOST) |
| **Stress reduction factor (`RFTRA`)** | **learned NN** |

In standard WOFOST, `RFTRA` represents a transpiration reduction factor:
a scalar that reduces gross assimilation under water-limited conditions.
Strictly speaking, it is intended to capture water stress only.

In this tutorial, however, we deliberately use `RFTRA` more broadly as a
generic stress gate on assimilation. The `_PP` (potential production) variant
of WOFOST does not model nutrient stress, and therefore systematically
overpredicts growth on nitrogen-limited plots. Instead of implementing the full
water- and nitrogen-limited production variants, we train a small neural network
to infer an effective daily stress factor directly from weather and treatment
context.

The physics still governs phenology, carbon allocation, and crop growth
dynamics; the neural network only modulates how much carbon is fixed each day.

**Disclaimer** This is admittedly a modelling shortcut — we are “misusing” `RFTRA` beyond its
original physiological interpretation — but it is defensible because RFTRA
acts multiplicatively on gross carbon assimilation. From the model’s perspective, both drought stress and nitrogen stress ultimately reduce canopy assimilation, even if the underlying physiological mechanisms differ.

**Activity (discuss with a neighbour).** What other parts of a crop model might
be good candidates for a hybrid replacement? What are the trade-offs of
replacing more physics with NN vs. less?


## 2. Run me first (Colab setup)

Run the collapsed setup cells below (click **Show code** on Colab to expand any of
them). They install diffwofost, download field data + pre-trained weights +
`tutorial_utils` helpers, and convert the weather files.

The hybrid NN classes (`StressNN`, `NNStressFactor`) are defined in **§5.1** once
we have introduced the architecture — not here.

After setup finishes, use **Runtime → Run all** (about 8–10 min on a free Colab
CPU runtime with pretrained weights).


In [ ]:
#@title Install diffwofost { display-mode: "form" }
#@markdown §2 — pinned git install.

!pip install -q "diffwofost @ git+https://github.com/WUR-AI/diffWOFOST@0a4d4a3b6682"


In [ ]:
#@title Imports and global settings { display-mode: "form" }
#@markdown §2 — libraries, paths, float64 on CPU.

import copy
import warnings
import sys
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import openpyxl
from openpyxl import Workbook
from pcse.base import ParameterProvider
from pcse.input import (
    ExcelWeatherDataProvider,
    YAMLAgroManagementReader,
    YAMLCropDataProvider,
    WOFOST72SiteDataProvider,
)
from pcse.util import DummySoilDataProvider
from pcse.traitlets import Instance

# diffwofost @ main (commit 0a4d4a3) — physics engine, Configuration
#   with CROP_COMPONENTS support (post-v0.4.0), classic_waterbalance.
# StressNN and NNStressFactor are NOT on PyPI yet; we defined in §5.1.
from diffwofost.physical_models.config import ComputeConfig, Configuration
from diffwofost.physical_models.crop.wofost72 import Wofost72
from diffwofost.physical_models.engine import Engine
from diffwofost.physical_models.soil.classic_waterbalance import WaterbalancePP
from diffwofost.physical_models.crop.evapotranspiration import Evapotranspiration

warnings.filterwarnings("ignore", message="To copy construct from a tensor.*")
ComputeConfig.set_device("cpu")
ComputeConfig.set_dtype(torch.float64)

# Colab puts everything under /content. These paths line up with what the
# data-fetch cell below produces, and what the rest of the notebook expects.
data_dir      = Path("/content/data")
data_temp_dir = Path("/content/data_temp")
print(f"data_dir      = {data_dir}")
print(f"data_temp_dir = {data_temp_dir}")


### 2.1 Download data and pre-trained weights

Three sources:

1. **This tutorial repo's `data/` directory** — three field-trial files
   (~900 KB) plus `models_bundle.zip` with the pre-trained `stress_nn_random.pt`
   (the hybrid) and `pure_lstm_random.pt` (the pure-ML reference).

   The field-trial files are mirrored verbatim from the
   [Harvard Dataverse dataset by Ten Den et al. (2024)](https://doi.org/10.7910/DVN/1LC6W7),
   licensed CC BY-NC-SA 4.0 (see [DATA_LICENSE.md](https://github.com/ronvree/diffwofost-tutorial/blob/master/DATA_LICENSE.md)).
   The model weights are derivative works of the same data and ship
   under the same licence.
2. **PCSE stock files** — config, crop YAML, and an agromanagement
   template, pulled from `ajwdewit/pcse` and `ajwdewit/pcse_notebooks`
   (Apache-2.0).
3. **Tutorial helper modules** — this repo's `tutorial_utils/*.py`,
   pulled into `/content/tutorial_utils/` so the rest of the notebook
   can `from tutorial_utils import ...`.


In [ ]:
#@title Download data and helpers { display-mode: "form" }
#@markdown §2 — field data, models, PCSE configs, tutorial_utils.

for d in [
    data_temp_dir,
    data_temp_dir / "trained_models",
    data_dir / "conf",
    data_dir / "crop",
    data_dir / "agro",
]:
    d.mkdir(parents=True, exist_ok=True)

# 1. Field-trial data + pre-trained models — this tutorial repo's data/.
#    Field data: Ten Den et al. (2024), CC BY-NC-SA 4.0, mirrored from
#    Harvard Dataverse (doi:10.7910/DVN/1LC6W7). Models: derivative works,
#    same licence. See DATA_LICENSE.md.
TUTORIAL_DATA = "https://raw.githubusercontent.com/ronvree/diffwofost-tutorial/master/data"
for name in [
    "Plotspecific_processed.csv",
    "Weatherfile_lelystad.xlsx",
    "Weatherfile_vredepeel.xlsx",
]:
    dest = data_temp_dir / name
    if not dest.exists():
        print(f"Downloading {name}...")
        urlretrieve(f"{TUTORIAL_DATA}/{name}", dest)

models_dir = data_temp_dir / "trained_models"
zip_path = models_dir / "models_bundle.zip"
if not (models_dir / "stress_nn_random.pt").exists():
    print("Downloading pre-trained models...")
    urlretrieve(f"{TUTORIAL_DATA}/models_bundle.zip", zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(models_dir)
    zip_path.unlink()

# 2. PCSE stock files (Apache-2.0)
PCSE_URL = "https://raw.githubusercontent.com/ajwdewit/pcse/master/pcse/conf"
NB_URL   = "https://raw.githubusercontent.com/ajwdewit/pcse_notebooks/master/data"
for url, dest in [
    (f"{PCSE_URL}/Wofost72_PP.conf",     data_dir / "conf" / "Wofost72_PP.conf"),
    (f"{NB_URL}/crop/crops.yaml",        data_dir / "crop" / "crops.yaml"),
    (f"{NB_URL}/crop/potato.yaml",       data_dir / "crop" / "potato.yaml"),
    (f"{NB_URL}/agro/AGMT_C2_2020.agro", data_dir / "agro" / "AGMT_C2_2020.agro"),
]:
    if not dest.exists():
        print(f"Downloading {dest.name}...")
        urlretrieve(url, dest)

# 3. The 2019 agromanagement file is a one-line variant of the 2020 template
#    (different year). Generate it inline rather than maintaining a copy.
agro_2019 = data_dir / "agro" / "AGMT_C2_2019.agro"
if not agro_2019.exists():
    agro_2019.write_text(
        "Version: 1.0\n"
        "AgroManagement:\n"
        "- 2019-04-20:\n"
        "    CropCalendar:\n"
        "        crop_name: 'potato'\n"
        "        variety_name: 'Potato_C2_C5'\n"
        "        crop_start_date: 2019-04-20\n"
        "        crop_start_type: 'sowing'\n"
        "        crop_end_date: 2019-10-31\n"
        "        crop_end_type: 'maturity'\n"
        "        max_duration: 300\n"
        "    TimedEvents:\n"
        "    StateEvents:\n"
    )

# 4. tutorial_utils helper modules — this repo, fetched as raw .py files.
TUTORIAL_UTILS_URL = "https://raw.githubusercontent.com/ronvree/diffwofost-tutorial/master/tutorial_utils"
utils_dir = Path("/content/tutorial_utils")
utils_dir.mkdir(parents=True, exist_ok=True)
for name in [
    "__init__.py",
    "data.py",
    "features.py",
    "losses.py",
    "training.py",
    "evaluation.py",
    "viz.py",
    "lstm.py",
]:
    dest = utils_dir / name
    if not dest.exists():
        print(f"Downloading tutorial_utils/{name}...")
        urlretrieve(f"{TUTORIAL_UTILS_URL}/{name}", dest)
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

# Convenient handles used by the rest of the notebook
conf_path = data_dir / "conf" / "Wofost72_PP.conf"
crop_path = data_dir / "crop"

print("\nAll data ready.")


In [ ]:
#@title Convert weather to PCSE format { display-mode: "form" }
#@markdown §2 — weather Excel → PCSE format.

from tutorial_utils.data import convert_weather_files

weather_paths = convert_weather_files(
    {
        "L": data_temp_dir / "Weatherfile_lelystad.xlsx",
        "V": data_temp_dir / "Weatherfile_vredepeel.xlsx",
    },
    data_temp_dir,
)
print(f"Weather files ready: {list(weather_paths.values())}")


## 3. Data inspection

The dataset is a multi-year potato field trial run at two Dutch sites:

- **Lelystad (L)** — clay soil, central Netherlands;
- **Vredepeel (V)** — sandy soil, southeast Netherlands.

The experimental design is a factorial:

```
2 years (2019, 2020) × 2 sites × 6 cultivars × 3 N-levels × 2 W-levels
168 plot-years (174 before dropping Shadow cultivar)
```

Each plot has ~7 destructive biomass samples taken over the growing season,
plus continuous weather records. Let's load it up and have a look under the hood.


In [ ]:
#@title Load field-trial data { display-mode: "form" }
#@markdown §3 — parse CSV, build plot index.

from tutorial_utils.data import load_observations, BIO_COLUMNS, ROOT_COLUMN

obs_df, PLOT_KEYS, plot_keys = load_observations(data_temp_dir / "Plotspecific_processed.csv")

print(f"Total plot-years (after Shadow drop): {len(PLOT_KEYS)}")
print()
print("Plot-years per (Year, Location, Cultivar):")
print(plot_keys.groupby(["Year", "Location", "Cultivar"]).size().unstack(fill_value=0))
print()
print("Per-variable observation counts:")
for col in BIO_COLUMNS + [ROOT_COLUMN]:
    print(f"  {col:<10} non-null: {obs_df[col].notna().sum()}")


## 4. Explore the data

Before fitting any model, a visual survey helps us sanity-check the dataset and
form intuition about what a model should be able to learn. Three quick views:

1. biomass / LAI trajectories per cultivar,
2. site-level comparison (Lelystad vs. Vredepeel),
3. treatment-effect preview (does N / W leave a signature?).


### 4.1 Biomass and LAI trajectories per cultivar

We expect leaves to peak around flowering, stems to plateau, tubers to rise
after flowering, and LAI to roughly follow the leaf curve. If anything looks
wildly off here, the data — not the model — is the first place to look.


In [ ]:
#@title Plot: biomass and LAI by cultivar { display-mode: "form" }
#@markdown §4.1

from tutorial_utils.viz import plot_cultivar_trajectories
plot_cultivar_trajectories(obs_df)


### 4.2 Site comparison

Lelystad vs. Vredepeel side by side. If the two sites differ enough that they
look like distinct populations, our model needs to be told which is which (the
site bit in the input features handles this).


In [ ]:
#@title Plot: site comparison { display-mode: "form" }
#@markdown §4.2

from tutorial_utils.viz import plot_site_comparison
plot_site_comparison(obs_df)


### 4.3 Treatment-effect preview

Do nitrogen level and irrigation treatment leave a visible mark on the
observations? If they do, the hybrid model has something to learn from them —
the NN gets a one-hot encoding of treatment as input.

**Activity (discuss with a neighbour).** Look at the two panels below. Which
treatment shows the cleaner ordering? Which observations look most likely to be
*driven* by stress, and which by genetics?


In [ ]:
#@title Plot: treatment-effect preview { display-mode: "form" }
#@markdown §4.3

from tutorial_utils.viz import plot_treatment_effects
plot_treatment_effects(obs_df)


## 5. The hybrid model

The hybrid we will build looks like this (simplified):

```
                +-------------------+      +-------------------+
   weather ---> | Phenology (DVS)   | ---> | Partitioning      |
   crop YAML    | (WOFOST physics)  |      | (WOFOST physics)  |
                +-------------------+      +-------------------+
                                                    |
                                                    v
                  +------------------+      +---------------------+
   weather ---->  | Stress NN        | -->  | Gross assimilation  |
   DVS, treatment | (learned RFTRA)  |      | (WOFOST × RFTRA)    |
                  +------------------+      +---------------------+
                                                    |
                                                    v
                                            organ biomass per day
                                                    |
                                            compare to observations
                                                    |
                                            loss --> backprop --> NN
```
**How it works.** In standard WOFOST, the daily stress reduction factor `RFTRA` is computed inside the evapotranspiration module. This factor represents how much crop assimilation should be reduced due to limited transpiration under water stress.
Under the `_PP` (potential production) configuration, however, no water stress
is simulated, and the model simply returns `RFTRA = 1.0` (no stress).

In this tutorial, we replace that evapotranspiration stress component with a
small neural-network module called `NNStressFactor`.

Each simulated day, the WOFOST engine calls this module to ask:
> “By how much should today’s gross assimilation be reduced?”


The replacement module:

1. assembles the day’s feature vector
(weather variables, DVS, and plot/treatment context),
2. feeds those inputs through a small multilayer perceptron (StressNN),
3. returns a sigmoid output as `RFTRA ∈ [0, 1]`.


Importantly, we use this learned stress factor not only for water stress, but
also implicitly for nutrient limitations such as nitrogen stress. This is a
modelling shortcut: physiologically, RFTRA was originally intended only for
transpiration-related stress. However, because it acts as a multiplicative
reduction on gross assimilation, it can also serve as an effective proxy for
other stressors that reduce canopy carbon fixation.

Because the entire simulation engine is implemented in PyTorch, every operation
remains differentiable. The seasonal loss (for example, yield prediction error)
therefore stays connected to the neural-network parameters through the complete
computational graph of the crop model.

Calling `loss.backward()` propagates gradients backward through the entire growing season and into the weights of `StressNN` — which is the central idea behind diffWOFOST.


### 5.1 The two plug-in components

`StressNN` is the tiny MLP that maps daily features → `RFTRA ∈ [0, 1]`.
`NNStressFactor` is the WOFOST evapotranspiration swap that calls it every
simulated day.

The implementations are included below so you can see exactly how the hybrid hook
works — skim the docstrings if you like, or jump to §6 for the input features.
They are embedded here because the diffwofost PyPI release does not yet ship this
NN integration (we install from a pinned commit in §2). Reference source:

- [`stress.py`](https://github.com/WUR-AI/diffWOFOST/blob/add-partitioning-sigmoid/src/diffwofost/ml_models/stress.py)
- [`evapotranspiration.py`](https://github.com/WUR-AI/diffWOFOST/blob/add-partitioning-sigmoid/src/diffwofost/ml_models/crop/evapotranspiration.py)


> 🔑 **Key idea**: `StressNN` is a tiny MLP (14 → 16 → 1, sigmoid). It is initialised so the *untrained* output is ~0.90 (mild stress) — that way training can learn deviations from the WOFOST baseline without starting from a random init that kills all growth.

In [ ]:
import torch

from diffwofost.physical_models.config import ComputeConfig


class StressNN(torch.nn.Module):
    """Tiny MLP mapping per-day features to an `RFTRA`-style stress factor.

    Architecture: `n_features → hidden_size → 1` with a `SiLU` activation in
    between and a `sigmoid` output. The output layer is biased at
    initialization toward `sigmoid(2.2) ≈ 0.90`, so an untrained network
    starts at a *mild stress* level (about 90% of PP). This avoids two
    pathologies: (a) collapsing to zero biomass before the optimizer can
    learn anything, and (b) starting in the saturated tail of the sigmoid
    where the gradient `sigmoid'(5) ≈ 0.007` would dampen all updates by
    two orders of magnitude. At bias=2.2 the gradient is `sigmoid'(2.2) ≈
    0.087` — roughly 13× larger — so the optimizer can actually move.

    Args:
        n_features (int): Length of the input feature vector.
        hidden_size (int): Width of the hidden layer. Defaults to 16.
        init_no_stress (bool): If True, biases the output toward ~0.90 at
            initialization (mild stress, with usable gradient). Defaults to
            True.
    """

    def __init__(self, n_features, hidden_size=16, init_no_stress=True):
        super().__init__()
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(n_features, hidden_size),
            torch.nn.SiLU(),
            torch.nn.Linear(hidden_size, 1),
        )
        if init_no_stress:
            with torch.no_grad():
                # bias=2.2 -> sigmoid(2.2) ~ 0.90, well outside the saturated
                # tail of the sigmoid so backprop gradients aren't dampened.
                # Output-layer weights are left at xavier defaults so the
                # network has full dynamic range to deviate from the bias.
                self.layers[-1].bias.fill_(2.2)
        self.to(device=ComputeConfig.get_device(), dtype=ComputeConfig.get_dtype())

    def forward(self, features):
        """Compute the stress factor from a feature vector.

        Args:
            features (torch.Tensor): 1-D tensor of input features.

        Returns:
            torch.Tensor: Scalar tensor in `[0, 1]` representing the stress
            factor (`RFTRA`).
        """
        out = self.layers(features)
        return torch.sigmoid(out).squeeze(-1)


> 🔑 **Key idea**: `NNStressFactor` is the PCSE-side shim that calls the NN each day and returns its output as `RFTRA`. **This is the seam where ML meets physics**: same interface as WOFOST's stock evapotranspiration class, so the engine swaps one for the other with no changes elsewhere.

In [ ]:
import torch
from pcse.traitlets import Instance

from diffwofost.physical_models.crop.evapotranspiration import Evapotranspiration


class NNStressFactor(Evapotranspiration):
    """Standard non-layered evapotranspiration with NN-overridden `RFTRA`.

    Runs the parent's `calc_rates` to populate `TRAMX`, `EVWMX`, `EVSMX`,
    `RFWS`, `RFOS`, then overrides `RFTRA` with `nn_model(features)` and
    recomputes `TRA = TRAMX × RFTRA`. The crop module reads `k.RFTRA` from
    the kiosk and applies it as `GASS = PGASS × k.RFTRA`, so the stress
    factor enters assimilation transparently.

    The feature builder is supplied as a callable rather than baked in,
    so the same engine class can be used with different feature sets.

    **Inputs from kiosk** (read indirectly by the parent's calc_rates and
    by the supplied `feature_builder`):

    | Name | Description                              |
    |------|------------------------------------------|
    | DVS  | Crop development stage                   |
    | LAI  | Leaf area index                          |
    | SM   | Soil moisture content                    |

    **Outputs to kiosk** (overridden):

    | Name  | Description                              |
    |-------|------------------------------------------|
    | RFTRA | NN-learned stress factor on assimilation |
    | TRA   | Crop transpiration rate (TRAMX × RFTRA)  |
    """

    nn_model = Instance(torch.nn.Module)
    feature_builder = Instance(object)

    def initialize(self, day, kiosk, parvalues, nn_model, feature_builder, shape=None):
        """Initialize using the parent's ET logic and attach the NN.

        Args:
            day: Start date of the simulation.
            kiosk (VariableKiosk): Variable kiosk of this PCSE instance.
            parvalues (ParameterProvider): Parameter provider for the ET module.
            nn_model (torch.nn.Module): Network mapping per-day feature
                vector to `RFTRA` in `[0, 1]`. Must return a scalar tensor.
            feature_builder (callable): Called daily as
                `feature_builder(day, drv, kiosk)` to produce a 1-D feature
                tensor for `nn_model`.
            shape (tuple | None): Target shape for state and rate variables.
        """
        super().initialize(day, kiosk, parvalues, shape=shape)
        self.nn_model = nn_model
        self.feature_builder = feature_builder

    def calc_rates(self, day=None, drv=None):
        """Compute ET rates with `RFTRA` replaced by the NN output."""
        super().calc_rates(day, drv)
        features = self.feature_builder(day, drv, self.kiosk)
        rftra = self.nn_model(features)
        if rftra.dim() > 0 and rftra.numel() == 1:
            rftra = rftra.reshape(())
        self.rates.RFTRA = rftra
        self.rates.TRA = self.rates.TRAMX * rftra


## 6. Building input features for the NN

The NN gets three groups of inputs:

**Per-plot context (constant over the season).**

| Field | Encoding | Dim |
|-------|----------|-----|
| Site | binary (Lelystad=0, Vredepeel=1) | 1 |
| Nitrogen level | numeric (N0=0.0, N1=0.3, N2=1.3) | 1 |
| Irrigation level | binary (W1=0, W2=1) | 1 |
| Cultivar | one-hot of [C1, C2, C3, C4, C5, C6] | 6 |

**Per-day weather features (computed once per site).**

| Field | Computed from | Dim |
|-------|---------------|-----|
| VPD (vapor pressure deficit) | `SVP(TMAX) − VAP` | 1 |
| TMAX (max temp) | weather file | 1 |
| 7-day rolling rainfall | weather file | 1 |
| IRRAD (radiation) | weather file (÷ 1e4) | 1 |

**Per-day crop-state features (read from the engine kiosk daily).**

| Field | Dim |
|-------|-----|
| DVS | 1 |

**Total NN input dim: 4 (weather) + 1 (DVS) + 9 (plot context) = 14.**



In [ ]:
#@title Feature encodings: plot context { display-mode: "form" }
#@markdown §6 — site, N, W, cultivar encodings.

from tutorial_utils.features import (
    SITE_INDEX, N_LEVEL_NUMERIC, W_LEVEL_INDEX, CULTIVARS, CULTIVAR_INDEX,
    PLOT_CONTEXT_DIM, make_plot_context_tensor,
)

print(f"Plot context dim: {PLOT_CONTEXT_DIM}")
print(f"Example (C2, N2, W2, Lelystad): {make_plot_context_tensor('C2', 'N2', 'W2', 'L')}")


In [ ]:
#@title Feature encodings: weather { display-mode: "form" }
#@markdown §6 — VPD, 7-day rolling rain, etc.

from tutorial_utils.features import (
    saturation_vp, build_weather_features, WEATHER_FEATURE_DIM,
)

WEATHER_FEATURES = build_weather_features(weather_paths)
print(f"Weather feature dim: {WEATHER_FEATURE_DIM}")
print(f"Lelystad features pre-computed for {len(WEATHER_FEATURES['L'])} days")
print(f"Vredepeel features pre-computed for {len(WEATHER_FEATURES['V'])} days")


Each simulated day, a **feature builder** gathers that day's weather, the current
development stage (`DVS`), and the plot's treatment info into one input vector for
`StressNN`. The engine calls it automatically at every time step — you don't
invoke it yourself (`tutorial_utils.features.PlotFeatureBuilder`).


In [ ]:
#@title Confirm NN input dimension { display-mode: "form" }
#@markdown §6 — 14 features total.

from tutorial_utils.features import PlotFeatureBuilder, N_FEATURES

print(f"Total NN input dim: {N_FEATURES}")


**Discussion question**

Crop stress is often **cumulative**: a week of low rain matters, not just today's
VPD. The hybrid stress NN is fed mostly same-day inputs (weather, `DVS`,
treatment), except for one explicit history term — **7-day rolling rainfall**.

1. Does that mix feel sufficient, or too memoryless for drought and N stress?
2. If stress should depend on what happened earlier in the season, what would you
   add? (Rolling windows? soil moisture from WOFOST? A recurrent network like the
   LSTM in §13?)


## 7. Engine setup

We use the standard `Wofost72_PP` stack — phenology, partitioning,
respiration, leaf dynamics, the works — but with a single targeted swap:
`evapotranspiration` → `NNStressFactor`. Everything else runs unchanged.


> 🔑 **Key idea**: this is where the NN slots *into* the WOFOST engine. Look for the line in `build_config` that overrides `cfg.CROP_COMPONENTS["evapotranspiration"]` with `NNStressFactor` — that single line is what turns a pure physics simulator into a hybrid model. Every other component (phenology, partitioning, leaf dynamics) is left untouched.


In [ ]:
#@title Engine setup { display-mode: "form" }
#@markdown §7 — WOFOST72_PP + NNStressFactor plug-in.

crop_data_provider = YAMLCropDataProvider(fpath=crop_path, force_reload=True)
parameter_provider = ParameterProvider(
    cropdata=crop_data_provider,
    soildata=DummySoilDataProvider(),
    sitedata=WOFOST72SiteDataProvider(WAV=10),
)

OUTPUT_VARS = ["DVS", "LAI", "TAGP", "WLV", "TWST", "TWSO", "TWRT", "RFTRA"]
base_pcse_config = Configuration.from_pcse_config_file(conf_path)


def build_config(et_class=None, et_kwargs=None):
    cfg = Configuration(
        CROP=Wofost72, SOIL=WaterbalancePP,
        AGROMANAGEMENT=base_pcse_config.AGROMANAGEMENT,
        OUTPUT_VARS=OUTPUT_VARS.copy(),
        SUMMARY_OUTPUT_VARS=list(base_pcse_config.SUMMARY_OUTPUT_VARS),
        TERMINAL_OUTPUT_VARS=list(base_pcse_config.TERMINAL_OUTPUT_VARS),
        OUTPUT_INTERVAL=base_pcse_config.OUTPUT_INTERVAL,
        OUTPUT_INTERVAL_DAYS=base_pcse_config.OUTPUT_INTERVAL_DAYS,
        OUTPUT_WEEKDAY=base_pcse_config.OUTPUT_WEEKDAY,
        model_config_file=base_pcse_config.model_config_file,
        description=base_pcse_config.description,
    )
    if et_class is not None:
        comp = {"class": et_class}
        if et_kwargs:
            comp.update(et_kwargs)
        cfg.CROP_COMPONENTS["evapotranspiration"] = comp
    return cfg


reference_config = build_config()  # default WOFOST72_PP, no NN

agro_paths = {
    2019: data_dir / "agro" / "AGMT_C2_2019.agro",
    2020: data_dir / "agro" / "AGMT_C2_2020.agro",
}


def results_to_tensors(results, var_names=OUTPUT_VARS):
    out = {"day": [pd.Timestamp(r["day"]) for r in results]}
    for v in var_names:
        out[v] = torch.stack([
            torch.as_tensor(r[v], dtype=ComputeConfig.get_dtype(),
                            device=ComputeConfig.get_device())
            for r in results
        ])
    return out


# Cache weather data providers (one per site)
WEATHER_DATA_PROVIDERS = {
    site: ExcelWeatherDataProvider(str(weather_paths[site]))
    for site in ["L", "V"]
}


def run_engine(config, year, location, weather):
    eng = Engine(config=config)
    eng.setup(copy.deepcopy(parameter_provider), weather, YAMLAgroManagementReader(agro_paths[year]))
    eng.run_till_terminate()
    return results_to_tensors(eng.get_output())


def run_plot_with_nn(plot, nn_model):
    feature_builder = PlotFeatureBuilder(
        WEATHER_FEATURES[plot.Location],
        make_plot_context_tensor(plot.Cultivar, plot.Nitrogen, plot.Irrigation, plot.Location),
    )
    cfg = build_config(NNStressFactor, et_kwargs={
        "nn_model": nn_model, "feature_builder": feature_builder,
    })
    return run_engine(cfg, plot.Year, plot.Location, WEATHER_DATA_PROVIDERS[plot.Location])


def run_plot_reference(plot):
    return run_engine(reference_config, plot.Year, plot.Location,
                      WEATHER_DATA_PROVIDERS[plot.Location])


print("Engine helpers ready.")


## 8. Prepare for training

Three preparatory pieces: a loss function, a reference-WOFOST baseline run (to
normalise the loss), and a train/test split.


### 8.1 Loss function

**The loss in plain English.** For every training plot we (1) run the hybrid simulation, (2) line up the simulated trajectory against the field-measured observation dates, (3) compute a normalised RMSE per observed variable (`WLV`, `TWST`, `TWSO`, `LAI`, plus a half-weighted `TWRT`), and (4) sum these into a single scalar — weighted so each variable contributes on a comparable scale.

*"Normalised"* means dividing by the mean of the observations, so the loss is in units of *fractional error* rather than the raw `kg/ha` (which would make `TWSO` dominate everything else).

The boring bookkeeping — matching simulated days to observation dates, building target tensors — lives in `tutorial_utils.losses`. What stays below is the pooled-loss math itself.


In [ ]:
#@title Loss function: pooled RMSE { display-mode: "form" }
#@markdown §8.1 — normalised RMSE summed over variables.

from tutorial_utils.losses import get_plot_observations, per_plot_loss

OBS_TO_PCSE = {"LeavesDW": "WLV", "StemDW": "TWST", "tubersDW": "TWSO", "LAI": "LAI"}
ROOT_PCSE = "TWRT"
VAR_BASE_WEIGHTS = {"WLV": 1.0, "TWST": 1.0, "TWSO": 1.0, "LAI": 1.0, "TWRT": 0.5}


def pooled_loss(plots, runner, weights):
    """Pooled normalised RMSE over plots and observed variables.

    For each plot: run `runner(plot)`, align predictions to observation dates,
    compute per-variable normalised RMSE (handled by `per_plot_loss`),
    weight-sum into a scalar.
    """
    total = torch.zeros((), dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device())
    n_used = 0
    all_diag = {k: [] for k in weights}
    per_plot = {}
    for p in plots:
        try:
            r = runner(p)
        except Exception as exc:
            print(f"  WARN: failed on plot {p}: {exc}")
            continue
        idx, tgt = get_plot_observations(
            p, r["day"], obs_df, BIO_COLUMNS, ROOT_COLUMN, OBS_TO_PCSE, ROOT_PCSE,
        )
        if idx is None:
            continue
        pl, diag = per_plot_loss(r, tgt, idx, weights)
        total = total + pl
        n_used += 1
        per_plot[(p.Year, p.Location, p.Plotnumber)] = r
        for k, v in diag.items():
            all_diag[k].append(v)
    avg_diag = {k: float(np.mean(v)) if v else float("nan") for k, v in all_diag.items()}
    return total / max(n_used, 1), avg_diag, per_plot, n_used


print("Loss machinery ready.")


### 8.2 Reference run: default WOFOST (no NN)

Before training the hybrid, we run **stock WOFOST72_PP** on every plot — no neural
network, no stress correction. That gives a **reference curve** for the fit plots
in §11 and the comparison in §12.

> ☕ **This cell takes a few minutes** — it simulates all plot-years on CPU.

> **Important caveat.** This WOFOST run is *uncalibrated* for this dataset (stock
> potato parameters, generic sowing schedule). It shows the **gap the hybrid is
> targeting**, not proof that hybrid beats a properly calibrated WOFOST. See the
> diffWOFOST `optimization_*.ipynb` notebooks for parameter-only calibration.
>
> The same run also **balances the training loss** across variables (leaves, stems,
> tubers, LAI, roots) so tuber biomass does not dominate — handled automatically
> in the code below.


In [ ]:
#@title Reference WOFOST run { display-mode: "form" }
#@markdown §8.2 — all plot-years, ~3–4 min on CPU.

print(f"Running reference WOFOST72_PP on {len(PLOT_KEYS)} plot-years...")
ref_loss_uniform, ref_diag, REFERENCE_PLOT_RESULTS, n_ref_used = pooled_loss(
    PLOT_KEYS, run_plot_reference, VAR_BASE_WEIGHTS,
)
print(f"  Reference plots successfully simulated: {n_ref_used}/{len(PLOT_KEYS)}")
print(f"  Reference pooled loss (uniform weights): {ref_loss_uniform.item():.4f}")
print()
print("Per-variable RMSE (averaged across plots):")
for k, v in ref_diag.items():
    print(f"  {k:<6} {v:.4f}")

NORMALIZED_WEIGHTS = {
    k: VAR_BASE_WEIGHTS[k] / max(ref_diag[k], 0.1) for k in VAR_BASE_WEIGHTS
}
print()
print("Normalised loss weights (used during training):")
for k, v in NORMALIZED_WEIGHTS.items():
    print(f"  {k:<6} {v:.3f}")


### 8.3 Train/test split

We use a simple random 80/20 split of the plot-years. This tests
*interpolation within the design space*: training and test see the same
cultivars, sites, and treatments — just different plots.

In [ ]:
#@title Train/test split { display-mode: "form" }
#@markdown §8.3 — random 80/20 of plot-years.

from tutorial_utils.data import split_random

train_plots, test_plots = split_random(PLOT_KEYS)
print(f"  train plot-years: {len(train_plots)}")
print(f"  test  plot-years: {len(test_plots)}")


## 9. Initialise the stress neural network

`StressNN` is a tiny MLP — 14 → 16 → 1 with a `SiLU` non-linearity and a
sigmoid output. With `init_no_stress=True`, the output bias is set so the
*untrained* network returns `RFTRA ≈ 0.90` (mild stress). This avoids two
pathologies:

- starting at `RFTRA = 0` would kill all biomass before the optimizer could
  learn anything;
- starting in the saturated tail of the sigmoid (e.g. `RFTRA = 0.99`) would
  shrink the gradient to almost nothing.

13× larger gradient at init = an optimizer that can actually move.


In [ ]:
torch.manual_seed(7)
stress_nn = StressNN(n_features=N_FEATURES, hidden_size=16, init_no_stress=True)
n_params = sum(p.numel() for p in stress_nn.parameters())
print(f"StressNN: {n_params} parameters")
print(stress_nn)


> 🔑 **Key idea**: every training step is *standard* SGD — `optimizer.zero_grad()` → forward (which runs a full season of WOFOST per plot) → `loss.backward()` → `optimizer.step()`. The novelty is that `backward()` propagates gradients **through the entire simulation** back into the NN parameters. From PyTorch's point of view the engine is just another differentiable module.


## 10. Train the model

Each training step does the following for every plot in the train set:

1. **Forward**: run the engine for the whole season. Every simulated day, the
   `NNStressFactor` component evaluates the NN to produce `RFTRA`, which
   reduces the day's gross assimilation.
2. **Compute loss**: extract simulated biomass at observation dates, compare to
   measurements, accumulate the normalised RMSE.
3. **Backward**: call `loss.backward()`. PyTorch propagates gradients through
   every operation in the engine — leaf growth, partitioning, root
   redistribution — and ends up at the NN parameters.
4. **Optimise**: Adam takes a step.

That's it. The engine looks like a normal PyTorch module from the optimizer's
point of view.

**Training is expensive** (each step = N engine runs × ~150 days each). To
keep this tutorial snappy we load a pre-trained checkpoint by default. Flip
`FORCE_RETRAIN = True` if you'd like to retrain from scratch (about 5–15 min
on CPU).


In [ ]:
#@title Train (or load) hybrid NN { display-mode: "form" }
#@markdown §10 — pretrained by default; FORCE_RETRAIN = True to retrain.

from tutorial_utils.training import EarlyStopper, try_load_checkpoint, save_checkpoint

FORCE_RETRAIN = False
MODEL_DIR = data_temp_dir / "trained_models"
MODEL_DIR.mkdir(exist_ok=True)
model_path = MODEL_DIR / "stress_nn_random.pt"

training_config = {"lr": 0.02, "max_steps": 60, "patience": 15, "min_delta": 5e-4}

saved = try_load_checkpoint(stress_nn, model_path) if not FORCE_RETRAIN else None
if saved is not None:
    training_run = saved["training_run"]
    print(f"Loading saved model from {model_path.name}")
    print(f"  Previously trained for {len(training_run['train_history'])} steps")
    print(f"  Saved train loss: {training_run['train_history'][-1]:.4f}")
    print(f"  Saved test  loss: {training_run['test_history'][-1]:.4f}")
    print(f"  (Set FORCE_RETRAIN = True above to retrain.)")
else:
    print("Training from scratch — this will take a while...")
    optimizer = torch.optim.Adam(stress_nn.parameters(), lr=training_config["lr"])
    stopper = EarlyStopper(training_config["patience"], training_config["min_delta"], stress_nn)
    train_history, test_history, diag_history = [], [], []

    def runner(plot):
        return run_plot_with_nn(plot, stress_nn)

    # The SGD loop: forward through the engine, backward through the engine,
    # step. Autograd takes care of propagating gradients across the whole
    # simulation back into the NN parameters.
    for step in range(training_config["max_steps"]):
        optimizer.zero_grad()
        train_loss, train_diag, _, _ = pooled_loss(train_plots, runner, NORMALIZED_WEIGHTS)
        train_history.append(train_loss.detach().cpu().item())
        diag_history.append(train_diag)

        with torch.no_grad():
            test_loss, _, _, _ = pooled_loss(test_plots, runner, NORMALIZED_WEIGHTS)
        test_history.append(test_loss.detach().cpu().item())

        if step % 2 == 0:
            print(f"  step {step:03d} | train={train_history[-1]:.4f} test={test_history[-1]:.4f}")

        stopper.update_best(train_history[-1], step, stress_nn)

        train_loss.backward()
        torch.nn.utils.clip_grad_norm_(stress_nn.parameters(), max_norm=1.0)
        optimizer.step()

        if stopper.should_stop(step):
            print(f"  early stopping at step {step}")
            break

    stopper.restore_best(stress_nn)
    training_run = {
        "train_history": train_history,
        "test_history":  test_history,
        "diag_history":  diag_history,
    }
    save_checkpoint(stress_nn, model_path, training_run=training_run, n_features=N_FEATURES)
    print(f"Saved trained model to {model_path}")

# Final evaluation — collect simulation outputs for every plot (train + test)
with torch.no_grad():
    final_train_loss, final_train_diag, train_results, _ = pooled_loss(
        train_plots, lambda p: run_plot_with_nn(p, stress_nn), NORMALIZED_WEIGHTS,
    )
    final_test_loss, final_test_diag, test_results, _ = pooled_loss(
        test_plots, lambda p: run_plot_with_nn(p, stress_nn), NORMALIZED_WEIGHTS,
    )

print()
print(f"FINAL: train loss={final_train_loss.item():.4f}  test loss={final_test_loss.item():.4f}")


## 11. Inspect model performance

We inspect five things:

1. **Loss curves** — did training converge? Does test track train?
2. **Per-plot fits** — does the model recover the shape of the trajectories?
3. **Learned stress factor per plot** — what does `RFTRA(t)` look like for a few plots?
4. **Stress profile by cultivar** — does the NN learn cultivar-specific stress?
5. **Stress profile by nitrogen treatment** — does it learn the N effect?


### 11.1 Loss curves


In [ ]:
#@title Plot: training loss curves { display-mode: "form" }
#@markdown §11.1

from tutorial_utils.viz import plot_loss_curves
plot_loss_curves(training_run)


### 11.2 Per-plot fits

For one plot per cultivar in the test set, we overlay:

- the default WOFOST72_PP simulation (solid),
- the trained hybrid simulation (dashed),
- the actual field observations (black dots).


In [ ]:
#@title Plot: per-plot model fits { display-mode: "form" }
#@markdown §11.2

from tutorial_utils.viz import select_representative_plots, plot_per_plot_fits

display_plots = select_representative_plots(test_plots, CULTIVARS, n_per_cultivar=1)
if not display_plots:
    display_plots = select_representative_plots(train_plots, CULTIVARS, n_per_cultivar=1)
print(f"Showing fits for {len(display_plots)} plots (one per cultivar in the test set)")

plot_per_plot_fits(display_plots, train_results, test_results, REFERENCE_PLOT_RESULTS, obs_df)


### 11.3 Learned stress factor over the season

`RFTRA(t)` is what the NN actually produces. A value of 1.0 means "no stress",
a value of 0.0 means "complete shutdown of carbon fixation". Reasonable
trajectories should sit between, dipping during the parts of the season where
the model needs to reduce biomass to match observations.


In [ ]:
#@title Plot: learned RFTRA trajectories { display-mode: "form" }
#@markdown §11.3

from tutorial_utils.viz import plot_rftra_per_plot
plot_rftra_per_plot(display_plots, train_results, test_results)


### 11.4 Stress profile by cultivar

Per-plot trajectories show us *one* learned curve at a time, but they don't
answer the question we really care about: *did the NN learn that different
cultivars need different stress profiles?* The cultivar one-hot is an input
feature, but nothing forces the NN to use it.

To check, we collapse the time axis from calendar dates to **DVS** (development
stage, 0 = emergence, 1 = anthesis, 2 = maturity), bin `RFTRA` into DVS bins,
and average across all plots of each cultivar (pooled across N, W, year, site).
If the cultivars come out as visibly different curves, the NN used the cultivar
input meaningfully. If they're nearly identical, it didn't — and your one-hot
embedding wasn't worth the dimensions.


In [ ]:
#@title Plot: RFTRA profile by cultivar { display-mode: "form" }
#@markdown §11.4

from tutorial_utils.evaluation import aggregate_rftra_by_dvs, DVS_CENTERS
from tutorial_utils.viz import plot_rftra_by_cultivar

cultivar_rftra_by_dvs = aggregate_rftra_by_dvs(
    train_plots + test_plots,
    [train_results, test_results],
    lambda p: p.Cultivar,
)
plot_rftra_by_cultivar(cultivar_rftra_by_dvs, CULTIVARS, DVS_CENTERS)


### 11.5 Stress profile by nitrogen treatment

Same DVS-binned aggregation as 11.4, but split by **N level** (N0 / N1 / N2)
instead of cultivar — pooled over all cultivars, sites, and W treatments. The
nitrogen treatment is encoded as a scalar input to the NN (N0=0.0, N1=0.3,
N2=1.3), so this plot is the cleanest test of whether the NN actually uses
that input.

Physiologically we'd expect:

- **N0** (no fertiliser) — most N-limited, so largest stress, lowest RFTRA;
- **N1** (~30% advised) — intermediate;
- **N2** (~130% advised) — least N-stressed, RFTRA closest to 1.

If the three curves come out essentially overlapping, the NN didn't
differentiate by N treatment — it absorbed the N-related residual into other
features (cultivar, weather) or just didn't fit it. That's important
information for *what to look at next*: it might mean the N encoding needs
more weight, the data lacks the signal, or the loss isn't sensitive enough to
treatment differences.


In [ ]:
#@title Plot: RFTRA profile by N treatment { display-mode: "form" }
#@markdown §11.5

from tutorial_utils.evaluation import aggregate_rftra_by_dvs
from tutorial_utils.viz import plot_rftra_by_nitrogen

n_rftra_by_dvs = aggregate_rftra_by_dvs(
    train_plots + test_plots,
    [train_results, test_results],
    lambda p: p.Nitrogen,
)
plot_rftra_by_nitrogen(n_rftra_by_dvs, DVS_CENTERS)


**Activity (discuss with a neighbour).** Compare the two profile plots
above. Which axis (cultivar / N treatment) shows the cleaner separation? What
does that tell you about which input dimension the NN found most informative —
and how confident should you be that the NN's behaviour reflects *real
physiology* versus *spurious correlation* in this small dataset?

## 12. Reference comparison: default WOFOST

Below we line up the trained hybrid against the *uncalibrated* default
WOFOST72_PP on per-variable normalised RMSE. Before reading the bars, please
keep the framing in mind:

> ⚠️ **This is not an "is hybrid better than WOFOST?" benchmark.**
>
> The default WOFOST72_PP run here uses stock potato parameters with no
> cultivar-specific calibration and a generic sowing schedule for all plots.
> A serious comparison would (i) calibrate per-cultivar genetic parameters
> against the same training data, (ii) match the sowing/management to each
> plot, (iii) ideally use `Wofost72_WLP` so water balance is in play.
>
> What this bar chart *does* show is the **gap that the hybrid is targeting**:
> the difference between the uncalibrated PP trajectory and the observations.
> The hybrid closes most of that gap, so we can see that *the modelling
> approach works* — gradient descent through the engine successfully shapes
> the NN to fit the residuals. The bars are a *diagnostic*, not a leaderboard.

A train/test gap also matters here: small = good generalisation; large =
overfitting on the training plots.


In [ ]:
#@title Plot: hybrid vs WOFOST RMSE { display-mode: "form" }
#@markdown §12

from tutorial_utils.evaluation import per_plot_diag
from tutorial_utils.viz import plot_per_variable_rmse_bars

train_diag_per_plot = per_plot_diag(
    train_plots, train_results, list(NORMALIZED_WEIGHTS.keys()),
    obs_df, BIO_COLUMNS, ROOT_COLUMN, OBS_TO_PCSE, ROOT_PCSE,
)
test_diag_per_plot = per_plot_diag(
    test_plots, test_results, list(NORMALIZED_WEIGHTS.keys()),
    obs_df, BIO_COLUMNS, ROOT_COLUMN, OBS_TO_PCSE, ROOT_PCSE,
)
ref_diag_per_plot = per_plot_diag(
    train_plots + test_plots, REFERENCE_PLOT_RESULTS, list(NORMALIZED_WEIGHTS.keys()),
    obs_df, BIO_COLUMNS, ROOT_COLUMN, OBS_TO_PCSE, ROOT_PCSE,
)

variables = list(NORMALIZED_WEIGHTS.keys())
plot_per_variable_rmse_bars(variables, train_diag_per_plot, test_diag_per_plot, ref_diag_per_plot)


**Activity (discuss with a neighbour).** Which variables does the hybrid
close the gap on most? Which does it barely move? Where do you think the
remaining test-set error comes from — the NN's capacity, the feature set, or
genuinely uncalibrated parts of WOFOST that the NN can't reach?

## 13. Comparison vs. a pure-ML LSTM

To complete the picture we now train a **pure data-driven model with no physics
in the loop**: a small LSTM that maps per-day features → daily organ biomass.
This tells us how much of the hybrid's fit comes from the *engine* versus the
*data*.

We choose an LSTM (not just an MLP) because the hybrid model implicitly gets
weather *history* through the engine's state variables (`LAI`, `WLV`, `DVS`,
…). To make the pure-ML reference comparable, it needs a way to integrate past
weather too — that's what the LSTM's hidden state does.

**Architecture:**

```
Per-day input (14):
  - Weather:    VPD, TMAX, 7-day rolling rain, IRRAD               [4]
  - Time:       days_since_sowing / 200                            [1]
  - Treatment:  site, N level, W level, cultivar one-hot           [9]

       inputs (200, 14)
              │
              ▼
       LSTM (32 hidden, 1 layer)
              │
              ▼
       Linear → Sigmoid × output_scales
              │
              ▼
       4 daily organ predictions   (WLV, TWST, TWSO, LAI)
```

The LSTM has access to the *same* inputs as the hybrid — same weather, same
treatment context. The only difference is: no physics in the loop. So any
performance gap between the two is attributable to the inductive bias of the
WOFOST engine.

> ⚠️ **Same caveat as the WOFOST baseline.** This LSTM is a vanilla reference
> architecture, not a tuned state-of-the-art. The point of the comparison is
> *qualitative*: how stable / how data-hungry / how prone to overfitting is a
> black-box ML model versus a hybrid one on this size of dataset? Don't read
> these bars as "hybrid > LSTM in absolute terms" — read them as "physics
> contributes inductive bias that matters when you have ~140 training plots."


### 13.1 LSTM data preparation

Pre-build the per-day feature sequences (one per plot) and the
observation-index / target tensors. Doing this once up front means the training
loop just slices into prebuilt tensors.


In [ ]:
#@title LSTM data preparation { display-mode: "form" }
#@markdown §13.1 — feature sequences for pure-ML baseline.

from tutorial_utils.lstm import (
    prepare_pure_features, build_lstm_datasets, PureLSTM,
    LSTM_SEQ_LEN, LSTM_HIDDEN, PURE_OBS_VARS, PURE_OBS_TO_PCSE, SOWING_DATE_BY_YEAR,
)

PURE_N_FEATURES = WEATHER_FEATURE_DIM + 1 + PLOT_CONTEXT_DIM

pure_features_for_date, PURE_FEATURES_MEAN, PURE_FEATURES_STD = prepare_pure_features(
    train_plots, obs_df, WEATHER_FEATURES, WEATHER_FEATURE_DIM, make_plot_context_tensor,
)

# Re-use the same per-variable loss weights as the hybrid model so the two
# losses are directly comparable.
PURE_VARIABLE_WEIGHTS = torch.tensor(
    [NORMALIZED_WEIGHTS[v] for v in PURE_OBS_TO_PCSE],
    dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device(),
)

print("Pre-building LSTM sequences for every plot...")
lstm_data_train, lstm_data_test = build_lstm_datasets(
    train_plots, test_plots, pure_features_for_date, obs_df,
)
print(f"  train sequences: {len(lstm_data_train)} plots, each ({LSTM_SEQ_LEN}, {PURE_N_FEATURES})")
print(f"  test  sequences: {len(lstm_data_test)} plots, each ({LSTM_SEQ_LEN}, {PURE_N_FEATURES})")


### 13.2 Train (or load) the LSTM

The LSTM trains the same way as the hybrid — pooled normalised RMSE, Adam,
early stopping. To prevent overfitting on the small dataset, we (a) carve a
separate validation split out of the training plots so early stopping does not
peek at the test set, and (b) add a touch of weight decay. As with the hybrid,
we load a pre-trained checkpoint by default.


In [ ]:
#@title Train (or load) LSTM { display-mode: "form" }
#@markdown §13.2 — pretrained by default.

from tutorial_utils.lstm import train_pure_lstm, lstm_pooled_loss

lstm_model_path = MODEL_DIR / "pure_lstm_random.pt"

torch.manual_seed(23)
pure_lstm = PureLSTM(n_features=PURE_N_FEATURES, hidden=LSTM_HIDDEN)
print(f"PureLSTM: {sum(p.numel() for p in pure_lstm.parameters())} parameters")

lstm_run = train_pure_lstm(
    pure_lstm, lstm_data_train, lstm_data_test, PURE_VARIABLE_WEIGHTS,
    lstm_model_path, force_retrain=FORCE_RETRAIN, pure_n_features=PURE_N_FEATURES,
)

with torch.no_grad():
    final_lstm_train_loss, final_lstm_train_diag, lstm_train_preds = lstm_pooled_loss(
        pure_lstm, lstm_data_train, PURE_VARIABLE_WEIGHTS,
    )
    final_lstm_test_loss, final_lstm_test_diag, lstm_test_preds = lstm_pooled_loss(
        pure_lstm, lstm_data_test, PURE_VARIABLE_WEIGHTS,
    )

print()
print(f"Pure LSTM | final train loss = {final_lstm_train_loss.item():.4f}")
print(f"Pure LSTM | final test  loss = {final_lstm_test_loss.item():.4f}")


### 13.3 Three-way comparison

Three views, each telling a different story.

**(i) Per-variable test RMSE.** Default WOFOST (uncalibrated reference),
hybrid, and pure LSTM side by side on the held-out test set. The hybrid
typically lands in between — it doesn't have the LSTM's raw memorisation
capacity but it generalises more reliably (see view (iii)).


In [ ]:
#@title Plot: three-model test RMSE { display-mode: "form" }
#@markdown §13.3

from tutorial_utils.viz import plot_three_way_rmse_bars
plot_three_way_rmse_bars(PURE_OBS_TO_PCSE, ref_diag, final_test_diag, final_lstm_test_diag)


**(ii) Trajectories on a single plot.** Showing all three models on the
*same* sample plot makes the qualitative differences clear: the hybrid's
trajectories obey the engine's physics (smooth, physiologically plausible
curves with the right shape), while the LSTM produces whatever shape minimises
its loss — sometimes biologically odd kinks or wiggles, particularly near the
start of the season where it has little context.


In [ ]:
#@title Plot: three-model trajectories { display-mode: "form" }
#@markdown §13.3

from tutorial_utils.lstm import build_lstm_data
from tutorial_utils.viz import plot_three_way_trajectories

sample_plot_for_lstm = test_plots[0] if test_plots else train_plots[0]
sample_data = lstm_data_test.get(sample_plot_for_lstm) or lstm_data_train.get(sample_plot_for_lstm)
if sample_data is None:
    seq_sample, _, _ = build_lstm_data(sample_plot_for_lstm, pure_features_for_date, obs_df)
else:
    seq_sample = sample_data[0]

with torch.no_grad():
    lstm_traj = pure_lstm(seq_sample)[0].cpu().numpy()
sowing_for_sample = SOWING_DATE_BY_YEAR[sample_plot_for_lstm.Year]
lstm_dates = [sowing_for_sample + pd.Timedelta(days=d) for d in range(LSTM_SEQ_LEN)]

hybrid_results = {**train_results, **test_results}
plot_three_way_trajectories(
    sample_plot_for_lstm, lstm_traj, lstm_dates,
    REFERENCE_PLOT_RESULTS, hybrid_results, obs_df, LSTM_SEQ_LEN,
)


**(iii) Train vs. test gap — the headline result.** This is where the
inductive bias of the engine pays off. The LSTM tends to fit the training set
*very* well (it has the capacity to memorise the ~140 training plots) but its
test loss is often markedly higher — a sign of overfitting. The hybrid has a
much smaller train/test ratio because the physics in the loop constrains what
shapes its predictions can take in the first place.

If you re-train the LSTM with a different seed, you'll usually see a different
test-set loss; re-training the hybrid is far more reproducible. That
**stability / consistency advantage** is one of the main reasons to keep
physics in the loop on small datasets.


In [ ]:
#@title Plot: hybrid vs LSTM generalisation { display-mode: "form" }
#@markdown §13.3

from tutorial_utils.viz import plot_train_test_gap
plot_train_test_gap(
    final_train_loss.item(), final_test_loss.item(),
    final_lstm_train_loss.item(), final_lstm_test_loss.item(),
)


**Activity (discuss with a neighbour).** Look at the three views above.
On *training* data, which model wins? On *test* data? And if we would do spits by cultivar which would you expect to degrade more? Why?


> 🔑 **Key idea**: setting `requires_grad=True` on a parameter tensor and calling `.backward()` once on any simulated quantity gives you ∂(quantity)/∂(param) for **every** parameter in one pass. The engine being differentiable end-to-end means autograd tracks the dependency through the whole season — no finite-differencing, no manual derivation.


## 14. Bonus: differentiability for free

This is the bit that makes `diffwofost` *different* from a pure forward
simulator. Because every operation in the engine is implemented in PyTorch, we
can compute gradients of *any* simulated quantity with respect to *any*
parameter — using one forward pass and one backward pass.

To demonstrate, we ask: *how much does the final tuber yield (`TWSO`) change if
each of `SPAN`, `TSUM1`, or `TSUM2` were perturbed by one unit?*

In a non-differentiable simulator, you'd run hundreds of finite-difference
simulations. With autograd, you get all sensitivities at once.


In [ ]:
sample_plot = test_plots[0] if test_plots else train_plots[0]
print(f"Sample plot: {sample_plot.Cultivar}@{sample_plot.Location} "
      f"{sample_plot.Nitrogen}{sample_plot.Irrigation} {sample_plot.Year}")


def run_with_param_overrides(plot, nn, **scalar_overrides):
    fb = PlotFeatureBuilder(WEATHER_FEATURES[plot.Location],
                            make_plot_context_tensor(plot.Cultivar, plot.Nitrogen,
                                                     plot.Irrigation, plot.Location))
    cfg = build_config(NNStressFactor, et_kwargs={"nn_model": nn, "feature_builder": fb})
    pp = copy.deepcopy(parameter_provider)
    for name, value in scalar_overrides.items():
        pp.set_override(name, value, check=False)
    engine = Engine(config=cfg)
    engine.setup(pp, WEATHER_DATA_PROVIDERS[plot.Location],
                 YAMLAgroManagementReader(agro_paths[plot.Year]))
    engine.run_till_terminate()
    return results_to_tensors(engine.get_output())


span_t = torch.tensor(35.0, dtype=ComputeConfig.get_dtype(), requires_grad=True)
tsum1_t = torch.tensor(150.0, dtype=ComputeConfig.get_dtype(), requires_grad=True)
tsum2_t = torch.tensor(1550.0, dtype=ComputeConfig.get_dtype(), requires_grad=True)

results_grad = run_with_param_overrides(
    sample_plot, stress_nn,
    SPAN=span_t, TSUM1=tsum1_t, TSUM2=tsum2_t,
)
twso_final = results_grad["TWSO"][-1]
print(f"\nSimulated final TWSO = {twso_final.item():.1f} kg/ha")

# ONE backward pass yields gradients to ALL parameters at once
twso_final.backward()
print()
print("Gradients of final TWSO with respect to each parameter (autograd):")
print(f"  d(TWSO)/d(SPAN)  = {span_t.grad.item():+.2f} kg/ha per day")
print(f"  d(TWSO)/d(TSUM1) = {tsum1_t.grad.item():+.4f} kg/ha per (degC*day)")
print(f"  d(TWSO)/d(TSUM2) = {tsum2_t.grad.item():+.4f} kg/ha per (degC*day)")


Translated into "% change in final TWSO per typical-magnitude
perturbation", the sensitivities look like this:


In [ ]:
#@title Plot: yield parameter sensitivities { display-mode: "form" }
#@markdown §14

from tutorial_utils.viz import plot_sensitivity_bars

TYPICAL_PERTURB = {"SPAN": 5.0, "TSUM1": 50.0, "TSUM2": 100.0}
gradients = {
    "SPAN":  span_t.grad.item(),
    "TSUM1": tsum1_t.grad.item(),
    "TSUM2": tsum2_t.grad.item(),
}
twso_baseline = twso_final.item()
sensitivities_pct = {
    name: (gradients[name] * TYPICAL_PERTURB[name]) / twso_baseline * 100
    for name in gradients
}

plot_sensitivity_bars(sensitivities_pct, TYPICAL_PERTURB, twso_baseline)


**Why this matters.**

- **Targeted calibration.** The gradient tells you *which* parameter to tune
  first when residuals appear.
- **Sensitivity analysis at scale.** Same cost per plot, regardless of how
  many parameters you probe.
- **Optimisation through the engine.** "What sowing date maximises yield under
  these climate scenarios?" is now a smooth optimisation problem.
- **Variational data assimilation.** Gradient-based DA (e.g. 4D-Var) uses
  exactly this computational structure.

The hybrid stress NN is one application of `diffwofost`. Gradients are the
underlying capability.


## 15. Recap and what's next

We just walked through the full pipeline of a hybrid crop model:

1. **Loaded a real field-trial dataset** (168 plot-years (174 before dropping Shadow cultivar), two sites, six
   cultivars, three N levels, two W levels).
2. **Diagnosed the gap**: default (uncalibrated) WOFOST72_PP overpredicts
   biomass on water/N-stressed plots because it has no stress module.
3. **Replaced the stress component** with a tiny NN producing daily `RFTRA`.
4. **Trained the NN end-to-end** with gradient descent through the WOFOST
   engine.
5. **Inspected what the NN learned**: per-plot trajectories, per-cultivar and
   per-N-treatment stress profiles — checking that the NN actually uses its
   inputs.
6. **Compared against two reference points**: an uncalibrated WOFOST baseline
   (to see what the NN is correcting) and a pure-ML LSTM (to see what the
   engine's inductive bias contributes — especially the smaller train/test
   gap).
7. **Saw the bonus**: free gradients with respect to any parameter.

### Mental model: where does each model belong?

| Model | Strengths | Weaknesses |
|-------|-----------|------------|
| **Calibrated WOFOST** | Interpretable, transferable, mechanistic | Labour-intensive to fit; assumptions break |
| **Uncalibrated WOFOST (this notebook's reference)** | None — just a starting point | Systematic biases, especially under stress |
| **Pure LSTM** | High fit on training data | Unstable, brittle on held-out conditions |
| **Hybrid (this tutorial)** | Reliable shape from physics, NN closes the gap | NN is opaque; needs careful feature design |

### Where to go from here

- **Partitioning hybrid**:
  https://github.com/WUR-AI/diffWOFOST/blob/main/docs/notebooks/hybrid_partitioning_wofost72.ipynb
  applies the same idea to a *different* WOFOST component (carbon
  partitioning), useful for understanding what hybrid replacement can and
  can't do.
- **Pure calibration**: see https://github.com/WUR-AI/diffWOFOST/blob/main/docs/notebooks/optimization_phenology.ipynb (and friends) for parameter-only
  calibration (no NN) using the same differentiable engine — that's the
  apples-to-apples WOFOST comparison this notebook deliberately does not do.

### Caveats

- The trained NN is specific to this sowing schedule and weather window.
  Predictions outside the training distribution are unreliable.
- Cultivar-specific genetics (`SPAN`, `TSUM2`) are *not* calibrated here.
  Residuals related to leaf longevity and time-to-maturity remain visible.

That's it — happy hybrid modelling!
